1. Análisis de la estrategia de imputación
En la primera parte del trabajo práctico se analizó la posibilidad de imputar algunos valores faltantes utilizando estadísticas calculadas por clase (Outcome).
Responda las siguientes preguntas:
¿Sería posible utilizar esta misma estrategia dentro de un pipeline que deba procesar pacientes nuevos?
¿Qué información estará disponible cuando el modelo se utilice para realizar predicciones y cuál no?
En caso de que esta estrategia no sea adecuada, proponga una alternativa y justifique su elección.

No es posible usar esta estrategia ya que al ser una paciente nueva no se contaría con el dato de Outcome (es la clase a predecir), si se utilizaran datos del set de entrenamiento para imputar los datos faltantes de un paciente nuevo se produciría un information leakage

Estarán disponibles todos los atributos predictores. Lo que no se tendrá a mano es la variable objetivo (si la paciente tiene o no diabetes)

Se propone entonces utilizar una imputación global basada en la mediana mediante la clase SimpleImputer(), teniendo en cuenta que el pipeline que se ha sugerido en clase calcula las medianas usando solamente los datos de entrenamiento, que luego se aplican en el conjunto de entrenamiento, los de prueba y nuevas pacientes sin tener en consideración la etiqueta Outcome. 

2. Diseño del pipeline de preprocesamiento
Diseñe un pipeline que permita preparar automáticamente los datos para su posterior utilización por un modelo de aprendizaje automático.
El pipeline deberá contemplar, cuando corresponda:

-tratamiento de valores inválidos

-imputación de valores faltantes

-escalado de atributos numéricos

-transformación de atributos con distribuciones fuertemente sesgadas.

Recuerde que el valor cero debe interpretarse como faltante únicamente en las variables donde no es fisiológicamente válido. No deben modificarse ceros válidos, como Pregnancies=0 
Utilice una división 80/20, estratificada por Outcome, con random_state=42. La división debe realizarse antes de ajustar cualquier transformación. 
Justifique cada una de las decisiones tomadas.

In [2]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

df_diabetes = pd.read_csv('diabetes.csv', index_col = 0) #Datos cargados

train_set, test_set = train_test_split(df_diabetes, test_size = 0.2, stratify = df_diabetes["Outcome"], random_state = 42)

diabetes_train = train_set.drop("Outcome", axis=1) #Eliminamos la columna Outcome

diabetes_labels = train_set["Outcome"].copy() #Nos quedamos solo con la columna Outcome

#Reemplazamos por NaN cuando corresponda
cols_invalidas =["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"] #No pueden ser 0

diabetes_train[cols_invalidas] = diabetes_train[cols_invalidas].replace(0, np.nan) #Con esta linea se reemplazan los 0 por NaN en las columnas que, por razones fisiológicas, no pueden ser 0

#Pipeline para variables numéricas
num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy = "median")),
    ("standarize", StandardScaler()),
    ])

#Pipeline para variables sesgadas (realizamos transformación logarítmica)
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log1p, inverse_func= np.expm1, feature_names_out="one-to-one"), #Utilizamos log1p y no log común ya que podemos tener 0 y log(0) no está definido.
    StandardScaler()
)

Para la división entre el set de entrenamiento y el de prueba se utilizaron los parámetros sugeridos. Luego se dividieron las columnas entre los atributos predictores y el atributo objetivo.

Para manejar los valores nulos no fisiológicamente posibles se eligió rescatar aquellas categorias que cumplan con esta condición en una lista para poder separarlas fácilmente y utilizar funciones propias de la biblioteca pandas para reemplazar el 0 por NaN.

Por último se eligió trabajar con dos tipos de pipeline, uno para las clases no sesgadas, donde se imputa con la mediana y se estandariza y un segundo camino para las clases sesgadas, donde se realiza una transformación logarítmica para salventar el sesgo. Notar que se utiliza log1p y no log ya que pueden existir valores 0 posibles, si se utilizara log se daría el caso de log(0), que no está definido.

3. Implementación del pipeline
Implemente el pipeline utilizando las herramientas de Scikit-Learn.
Cuando distintos atributos requieran tratamientos diferentes, utilice ColumnTransformer para aplicar transformaciones específicas a cada grupo de variables.
El pipeline deberá ajustarse (fit) únicamente utilizando el conjunto de entrenamiento.

In [5]:
#Definimos ColumnTransformer
from sklearn.compose import ColumnTransformer

preprocessing_diabetes = ColumnTransformer(
    [
       ("log", log_pipeline, ["Pregnancies", "Age", "DiabetesPedigreeFunction", "Insulin"],), #Columnas que sufren una transformación logaritmica
       ("num", num_pipeline, ["Glucose", "BloodPressure", "SkinThickness", "BMI"]) #Columnas que no precisan una transformación logaritmica
    ]
)

preprocessing_diabetes.fit(diabetes_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

4. Aplicación del pipeline
Utilice el pipeline implementado para obtener:
el conjunto de entrenamiento preparado
el conjunto de prueba preparado.

Luego responda:

¿Cuál es la diferencia entre los métodos fit(), transform() y fit_transform()?

¿Qué método debe utilizarse sobre el conjunto de entrenamiento?

¿Qué método debe utilizarse sobre el conjunto de prueba?

¿Por qué aplicar fit() sobre el conjunto de prueba sería un error metodológico?

Justifique sus respuestas.

In [6]:
diabetes_test = test_set.drop("Outcome", axis = 1)
diabetes_test_labels = test_set["Outcome"].copy ()

diabetes_train_prepared = preprocessing_diabetes.transform(diabetes_train)

diabetes_test_prepared = preprocessing_diabetes.transform(diabetes_test)

print("Dimensiones de Entrenamiento preparado: ", diabetes_train_prepared.shape)
print("Dimensiones de Prueba preparado: ", diabetes_test_prepared.shape)

Dimensiones de Entrenamiento preparado:  (614, 8)
Dimensiones de Prueba preparado:  (154, 8)


El método fit() se utiliza para aprender y calcular parámetros estadísticos de los datos. No realiza alteraciones ni transformaciones sobre los mismos.

El método transform() transforma matemáticamente las columnas utilizando los parámetros previamente calculados por fit(). No aprende ningún parámetro nuevo de los datos que recibe.

El método fit_transform() es un método optimizado que concatena fit y luego transform sobre un mismo conjunto.



Para el de entrenamiento debe usarse fit() y luego transform() o bien fit_transform(), esto para que los transformadores aprendan las reglas estadísticas a partir de los datos con los que el modelo aprende.

Para el de prueba solo se debe utilizar el método transform() debido que si se utilizara fit() se daría un error de data leakage, al calcular nuevas medias, medianas y errores estándar basadas en ese nuevo conjunto. 

5. Verificación del resultado
Una vez aplicado el pipeline:

-verifique que no existan valores faltantes en los datos procesados

-verifique que entrenamiento y prueba posean exactamente los mismos atributos

-obtenga los nombres de las variables generadas por el pipeline mediante get_feature_names_out() (cuando corresponda).

Explique por qué estas verificaciones son importantes antes de entrenar un modelo de aprendizaje automático.


In [10]:
#Verificación de valores faltantes
diabetes_train_prepared_df = pd.DataFrame( #pd.DataFrame permite reconstruir los DataFrames
    diabetes_train_prepared,
    columns = preprocessing_diabetes.get_feature_names_out(),
    index = diabetes_train.index
)

diabetes_test_prepared_df = pd.DataFrame(
    diabetes_test_prepared,
    columns = preprocessing_diabetes.get_feature_names_out(),
    index = diabetes_test.index
)

#Para verificar si existen valores faltantes, se utiliza el metodo .isnull
nulos_entrenamiento  = diabetes_train_prepared_df.isnull().any(axis=1)
nulos_prueba = diabetes_test_prepared_df.isnull().any(axis=1)

print("Filas con nulos en Train Prepared: ", diabetes_train_prepared_df[nulos_entrenamiento])
print("Filas con nulos en Test Prepared: ", diabetes_test_prepared_df[nulos_prueba])


Filas con nulos en Train Prepared:  Empty DataFrame
Columns: [log__Pregnancies, log__Age, log__DiabetesPedigreeFunction, log__Insulin, num__Glucose, num__BloodPressure, num__SkinThickness, num__BMI]
Index: []
Filas con nulos en Test Prepared:  Empty DataFrame
Columns: [log__Pregnancies, log__Age, log__DiabetesPedigreeFunction, log__Insulin, num__Glucose, num__BloodPressure, num__SkinThickness, num__BMI]
Index: []


Esta salida demuestra dos cosas de relevancia.

Primero, se observa como se han imputado valores de forma correcta en ambos grupo. Empty DataFrame significa, en este caso, que no se ha encontrado en ellos alguna fila que contuviera en sus valores un valor nulo (NaN). Se corrobora entonces que el pipeline ha podido imputar de manera correcta en dichos casos.

También puede observarse que ambos grupos cuentan con ocho columnas, que son las propias que aportan información, o sea, se ha eliminado correctamente el índice Unnamed.

In [11]:
#Se verifica si train y test poseen los mismos atributos
dimensiones_train = diabetes_train_prepared_df.shape
dimensiones_test = diabetes_test_prepared_df.shape  #OBTENEMOS LAS DIMENSIONES

print("Dimensiones de Entrenamiento preparado:", dimensiones_train)
print("Dimensiones de Prueba preparado:", dimensiones_test)


print("Columnas del Set de Entrenamiento:")
print(list(diabetes_train_prepared_df.columns))


print("Columnas del Set de Prueba:")
print(list(diabetes_test_prepared_df.columns))

Dimensiones de Entrenamiento preparado: (614, 8)
Dimensiones de Prueba preparado: (154, 8)
Columnas del Set de Entrenamiento:
['log__Pregnancies', 'log__Age', 'log__DiabetesPedigreeFunction', 'log__Insulin', 'num__Glucose', 'num__BloodPressure', 'num__SkinThickness', 'num__BMI']
Columnas del Set de Prueba:
['log__Pregnancies', 'log__Age', 'log__DiabetesPedigreeFunction', 'log__Insulin', 'num__Glucose', 'num__BloodPressure', 'num__SkinThickness', 'num__BMI']


Se observa con el método .shape() que ambos sets cuentan con 8 columnas, cada una representando un atributo

Luego, se verifica que ambos sets constan con los mismos 8 atributos simplemente observando sus nombres.

También responde a la actividad siguiente donde se pide observar los nombres de las variables generadas.


Realizar estas verificaciones resulta de importancia ya que permite observar de manera rápida si se han preparado con cautela los sets del modelo. No hacerlo podría dejar errores tales como tener columnas faltantes o no tener las mismas columnas en ambos grupos